# Создание признаков

## Импорт библиотек

In [1]:
import sys
# Добавим папку проекта в список системных директорий, чтобы Python видел путь к папке rec_sys
sys.path.append("../../")

In [86]:
import numpy as np
import pandas as pd
from warnings import simplefilter

from rec_sys.utils.prepare_data import (
    convert_timespamp_to_datetime,
)
from rec_sys.utils.display_content import (
    display_nan_count,
)
from rec_sys.utils.constants import (
    TEST_SIZE,
)
from rec_sys.utils.prepare_data import (
    get_categories_levels_path,
)

pd.set_option("display.max_colwidth", None)

%matplotlib inline

simplefilter("ignore")

In [3]:
# Путь к папке с данными
DATA_PATH = "../../data"

## Загрузка данных

Описание исходных данных находится в основном файле [01_main.ipynb](./01_main.ipynb).

В файле с исследованием данных [02_eda.ipynb](./02_eda.ipynb) \
была создана таблица `events_advanced_data.csv` с дополнительными свойствами товаров.\
Используем ее для генерации признаков.

In [4]:
# Расширенная таблица с данными о событиях
events_advanced_data = pd.read_csv(f"{DATA_PATH}/events_advanced_data.csv")
# Признак даты переведем в тип datetime
events_advanced_data["event_datetime"] = pd.to_datetime(events_advanced_data["event_datetime"])

# Данные о структуре каталога
category_data = pd.read_csv(f"{DATA_PATH}/category_tree.csv")

# Данные о товарах
items_data = pd.concat([
    pd.read_csv(f"{DATA_PATH}/item_properties_part1.csv"),
    pd.read_csv(f"{DATA_PATH}/item_properties_part2.csv")
])
# Переведем признак timestamp в формат с отображением даты и времени
items_data = convert_timespamp_to_datetime(items_data)

Посмотрим на первые строки таблиц.

In [5]:
events_advanced_data.head(3)

,visitorid,event,itemid,transactionid,event_datetime,date,year,month,day,day_of_week,hour,minute,is_transaction,time_of_day,update_stage,available,price,categoryid,is_price_0,root_categoryid
0,693516,addtocart,297662,NaN,2015-05-03 03:00:04.384,2015-05-03,2015,5,3,6,3,0,False,3-7,0,1.0,14280.0,NaN,False,NaN
1,829044,view,60987,NaN,2015-05-03 03:00:11.289,2015-05-03,2015,5,3,6,3,0,False,3-7,0,0.0,204120.0,NaN,False,NaN
2,652699,view,252860,NaN,2015-05-03 03:00:13.048,2015-05-03,2015,5,3,6,3,0,False,3-7,0,NaN,NaN,NaN,False,NaN


In [6]:
category_data.head(3)

,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0


In [7]:
items_data.head(3)

,itemid,property,value,event_datetime
0,317951,790,n32880.000,2015-05-10 03:00:00
1,422842,480,1133979,2015-05-10 03:00:00
2,310185,776,103591,2015-05-10 03:00:00


Далее составим две таблицы.
- Таблицу с данными о пользователях
- Таблицу с данными о товарах

Создавать их будем из тренировочных данных.\
Поэтому сначала разделим выборку на обучающую и тестовую.

## Разделение выборки на обучающую и тестовую

В качестве тестовой выборки возьмем примерно 30% от конца данных.\
Посмотрим, на какую дату приходится начало этого диапазона.

In [8]:
# Общее количество записей в таблице с событиями
events_data_size = len(events_advanced_data)

# Количество записей для тестовой выборки
events_test_size = round(events_data_size * TEST_SIZE)

# Выведем начало этого хвоста и посмотрим на его первую дату
events_advanced_data["event_datetime"].tail(events_test_size).head(1)

1929271   2015-08-02 23:31:03.263
Name: event_datetime, dtype: datetime64[ns]

Последние 30 % записей о событиях начинаются со 2 августа.\
Для возможной дальнейшей простоты расчетов возьмем тестовую выборку со времени `2015-08-02 03:00:00`,\
которое соответствует началу 11-й итерации обновления данных каталога \
(это мы выяснили при исследовании данных в файле [02_eda.ipynb](./02_eda.ipynb)).

In [9]:
SPLIT_DATE = pd.Timestamp("2015-08-02 03:00:00")
mask_before_split_date = events_advanced_data["event_datetime"] < SPLIT_DATE

events_train_data = events_advanced_data[mask_before_split_date] 
events_test_data = events_advanced_data[~mask_before_split_date]

print("Train part:", round(len(events_train_data) / events_data_size * 100, 2), "%")
print("Test part:", round(len(events_test_data) / events_data_size * 100, 2), "%")

Train part: 69.57 %
Test part: 30.43 %


Данные таблицы `items_data` представляют из себя поэтапное обновление данных каталога товаров\
за определенный период, который примерно соответствует данным из таблицы с событиями.

Исходных данных всех товаров у нас нет. \
Есть только некоторые данные по тем товарам, которые обновлялись в указанный промежуток.\
Как мы увидим ниже, есть некоторое несоответствие множества обновленных товаров\
и множества товаров, которые встречаются в таблице событий.

Можно предположить, что в реальных условиях построения рекомендательной системы\
у нас все же были бы данные обо всем каталоге.\
Так как у компаний обычно есть данные об их товарах.

Поэтому для бОльшей полноты картины возьмем все данные из таблицы `items_data`.\
И для простоты расчетов оставим только последюю информацию по обновлению каждого свойства каждого товара.

Потому что разделение на тренировочную и тестовую выборку в данном случае не приближает условия задачи к реальной.

In [10]:
items_last_data = items_data.groupby(by=["itemid", "property"]).tail(1)

items_last_data.head(2)

,itemid,property,value,event_datetime
1,422842,480,1133979,2015-05-10 03:00:00
2,310185,776,103591,2015-05-10 03:00:00


## Создание таблицы с данными пользователей

Из тренировочной таблицы `events_train_data` соберем таблицу с данными пользователей.

In [11]:
events_train_data.head(3)

,visitorid,event,itemid,transactionid,event_datetime,date,year,month,day,day_of_week,hour,minute,is_transaction,time_of_day,update_stage,available,price,categoryid,is_price_0,root_categoryid
0,693516,addtocart,297662,NaN,2015-05-03 03:00:04.384,2015-05-03,2015,5,3,6,3,0,False,3-7,0,1.0,14280.0,NaN,False,NaN
1,829044,view,60987,NaN,2015-05-03 03:00:11.289,2015-05-03,2015,5,3,6,3,0,False,3-7,0,0.0,204120.0,NaN,False,NaN
2,652699,view,252860,NaN,2015-05-03 03:00:13.048,2015-05-03,2015,5,3,6,3,0,False,3-7,0,NaN,NaN,NaN,False,NaN


Для каждого пользователя сформируем следующие признаки.

- `visitorid` пользователя.
- Общее количество действий каждого типа  (столбец `event`).
- Максимальное, минимальное и среднее количество товаров в одной покупке \
(одну покупку будем определять по уникальному номеру транзакции в `transactionid`).
- Количество действий каждого типа  (`event`) в каждый день недели (`day_of_week`).
- Количество действий каждого типа (`event`) в каждое время дня (`time_of_day`).
- Количество купленных товаров с `available == 0`.
- Средняя и медианная стоимость всех купленных товаров (`price`).
- Количество купленных товаров с ценой 0 (`is_price_0`).
- Количество купленных товаров в каждой корневой категории (`root_categoryid`).

### Исследование пересечения данных о пользователях в тренировочной и тестовой выборках

In [12]:
# Множество пользователей в обеих выборках
userids_in_events = set(events_advanced_data["visitorid"].unique())

# Множество уникальных пользователей в тренировочной выборке
userids_in_train_events = set(events_train_data["visitorid"].unique())

# Множество уникальных пользователей в тестовой выборке
userids_in_test_events = set(events_test_data["visitorid"].unique())

print(f"Количество уникальных пользователей в обеих выборках: {len(userids_in_events)}")
print(
    "Количество уникальных пользователей в тренировочной выборке:",
    len(userids_in_train_events)
)
print(
    "Количество уникальных пользователей в тестовой выборке:",
    len(userids_in_test_events)
)

Количество уникальных пользователей в обеих выборках: 1407580
Количество уникальных пользователей в тренировочной выборке: 972459
Количество уникальных пользователей в тестовой выборке: 465137


In [13]:
# Множество пользователей, которые есть в тестовой выборке, но отсутствуют в тренировочной
userids_in_test_events_and_not_train = userids_in_test_events - userids_in_train_events

print(
    "Количество пользователей из тестовой выборки, которых нет в тренировочной:",
    len(userids_in_test_events_and_not_train)
)
print(
    "Процент пользователей из тестовой выборки, которых нет в тренировочной:",
    round(len(userids_in_test_events_and_not_train) / len(userids_in_test_events) * 100, 2),
    "%"
)

Количество пользователей из тестовой выборки, которых нет в тренировочной: 435121
Процент пользователей из тестовой выборки, которых нет в тренировочной: 93.55 %


> То есть, у нас почти 94% пользователей, которых модель на этапе погноза увидит впервые.\
Мы будем о них знать только время захода на сайт.\
И у нас не будет данных ни об их взаимодействии с товарами, \
ни об их похожести на других пользователей.\
То есть, у нас довольно мало данных для построения прогноза.

In [14]:
# Множество пользователей, которые есть в обеих выборках (тренировочной  и тестовой)
userids_in_train_and_test_events = userids_in_test_events & userids_in_train_events

print(
    "Количество общих пользователей в тренировочной и тестовой выборках:",
    len(userids_in_train_and_test_events)
)
print(
    "Процент общих пользователей в тренировочной выборке:",
    round(len(userids_in_train_and_test_events) / len(userids_in_train_events) * 100, 2),
    "%"
)
print(
    "Процент общих пользователей в тестовой выборке:",
    round(len(userids_in_train_and_test_events) / len(userids_in_test_events) * 100, 2),
    "%"
)

Количество общих пользователей в тренировочной и тестовой выборках: 30016
Процент общих пользователей в тренировочной выборке: 3.09 %
Процент общих пользователей в тестовой выборке: 6.45 %


> Таким образом, модель при обучении увидит довольно мало данных из тех, что она получит при прогнозе.\
У нас будет много холодных пользователей, о которых мы будем знать только время захода на сайт.

Далее соберем таблицу с данными о пользователях на основе тренировочной выборки.

### Общее количество действий каждого типа (столбец `event`)

In [15]:
users_data = events_train_data.pivot_table(
    index="visitorid",
    columns="event",
    values="is_transaction",
    aggfunc="sum",
    fill_value=0,
) \
.reset_index() \
.rename(columns={
    "view": "views_count",
    "addtocart": "adds_count", 
    "transaction": "trans_cout", 
})

users_data.sort_values("trans_cout", ascending=False).head(3)

event,visitorid,adds_count,trans_cout,views_count
794596,1150086,0,390,0
473028,684514,0,189,0
53197,76757,0,185,0


In [16]:
display_nan_count(users_data)

**Количество пропусков: 0**

### Максимальное, минимальное и среднее количество товаров в одной покупке 
(одну покупку будем определять по уникальному номеру транзакции в `transactionid`)

In [17]:
# Создадим таблицу пользователей с количеством товаров в каждой транзакции
users_transactions_volume = events_train_data.pivot_table(
    index=["visitorid", "transactionid"],
    values="is_transaction",
    aggfunc="sum"
) \
.reset_index() \
.rename(columns={"is_transaction": "items_count"}) 

users_transactions_volume.sort_values("items_count", ascending=False).head(3)

,visitorid,transactionid,items_count
6451,761482,7063.0,31
6450,761482,765.0,28
8458,974226,2753.0,23


In [18]:
# Возьмем минимальное, среднее и максимальное количество товаров 
# (купленных за один раз) для каждого пользователя
users_transactions_agg = users_transactions_volume.pivot_table(
    index="visitorid",
    values="items_count",
    aggfunc=["min", "mean", "max"]
) 

# Переименуем колонки с мульти-индексом
users_transactions_agg.columns = users_transactions_agg.columns.to_flat_index()

users_transactions_agg = users_transactions_agg.reset_index().rename(columns={
    ('min', 'items_count'): 'min_items_in_trans',
    ('mean', 'items_count'): 'mean_items_in_trans',
    ('max', 'items_count'): 'max_items_in_trans',
})

users_transactions_agg.head(3)

,visitorid,min_items_in_trans,mean_items_in_trans,max_items_in_trans
0,419,1,1.0,1
1,539,1,1.0,1
2,964,1,1.0,1


In [19]:
# Объединим полученные данные с основной таблицей о пользователях
users_data = users_data.merge(
    users_transactions_agg,
    on="visitorid",
    how="left",
).fillna(0)

In [20]:
# Посмотрим на результат
users_data.sort_values("min_items_in_trans", ascending=False).head(3)

,visitorid,adds_count,trans_cout,views_count,min_items_in_trans,mean_items_in_trans,max_items_in_trans
526074,761482,0,59,0,28.0,29.5,31.0
568226,822310,0,16,0,16.0,16.0,16.0
391032,565903,0,15,0,15.0,15.0,15.0


In [21]:
display_nan_count(users_data)

**Количество пропусков: 0**

### Количество действий каждого типа (`event`) в каждый день недели (`day_of_week`)

In [22]:
users_events_by_days = events_train_data.pivot_table(
    index="visitorid",
    columns=["day_of_week", "event"],
    values="itemid",
    aggfunc="count",
    fill_value=0,
)

# Переименуем колонки с мульти-индексом
users_events_by_days.columns = users_events_by_days.columns.to_flat_index()

# Соберем словарь для переименования колонок
EVENT_SHORT_NAMES = {
    "addtocart": "adds",
    "transaction": "trans",
    "view": "views",
}
columns_renaming = dict()
for day in range(0, 7):
    for event in ["addtocart", "transaction", "view"]:
        columns_renaming[(day, event)] = f"{EVENT_SHORT_NAMES[event]}_in_day_{day}"

users_events_by_days = users_events_by_days.reset_index().rename(columns=columns_renaming)

users_events_by_days.head(3)

,visitorid,adds_in_day_0,trans_in_day_0,views_in_day_0,adds_in_day_1,trans_in_day_1,views_in_day_1,adds_in_day_2,trans_in_day_2,views_in_day_2,...,views_in_day_3,adds_in_day_4,trans_in_day_4,views_in_day_4,adds_in_day_5,trans_in_day_5,views_in_day_5,adds_in_day_6,trans_in_day_6,views_in_day_6
0,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,5,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,7,0,0,0,0,0,0,0,0,0,...,2,0,0,0,0,0,1,0,0,0


In [23]:
# Объединим полученные данные с основной таблицей о пользователях
users_data = users_data.merge(
    users_events_by_days,
    on="visitorid",
    how="left",
).fillna(0)

In [24]:
# Посмотрим на результат
users_data.sort_values("trans_in_day_0", ascending=False).head(3)

,visitorid,adds_count,trans_cout,views_count,min_items_in_trans,mean_items_in_trans,max_items_in_trans,adds_in_day_0,trans_in_day_0,views_in_day_0,...,views_in_day_3,adds_in_day_4,trans_in_day_4,views_in_day_4,adds_in_day_5,trans_in_day_5,views_in_day_5,adds_in_day_6,trans_in_day_6,views_in_day_6
794596,1150086,0,390,0,1.0,1.114286,5.0,61,51,687,...,699,94,82,900,35,30,471,29,24,344
53197,76757,0,185,0,1.0,1.193548,5.0,73,36,274,...,211,30,24,237,0,0,5,0,0,0
473028,684514,0,189,0,1.0,1.166667,5.0,35,35,298,...,382,18,14,311,18,10,94,23,17,81


In [25]:
display_nan_count(users_data)

**Количество пропусков: 0**

### Количество действий каждого типа (`event`) в каждое время дня (`time_of_day`)

In [26]:
users_events_by_time = events_train_data.pivot_table(
    index="visitorid",
    columns=["time_of_day", "event"],
    values="itemid",
    aggfunc="count",
    fill_value=0,
)

# Переименуем колонки с мульти-индексом
users_events_by_time.columns = users_events_by_time.columns.to_flat_index()

# Соберем словарь для переименования колонок
columns_renaming = dict()
for time in ["3-7", "7-12", "12-16", "16-22", "22-3"]:
    for event in ["addtocart", "transaction", "view"]:
        columns_renaming[(time, event)] = f"{EVENT_SHORT_NAMES[event]}_in_time_{time}"

users_events_by_time = users_events_by_time.reset_index().rename(columns=columns_renaming)

users_events_by_time.head(3)

,visitorid,adds_in_time_12-16,trans_in_time_12-16,views_in_time_12-16,adds_in_time_16-22,trans_in_time_16-22,views_in_time_16-22,adds_in_time_22-3,trans_in_time_22-3,views_in_time_22-3,adds_in_time_3-7,trans_in_time_3-7,views_in_time_3-7,adds_in_time_7-12,trans_in_time_7-12,views_in_time_7-12
0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,5,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,7,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0


In [27]:
# Объединим полученные данные с основной таблицей о пользователях
users_data = users_data.merge(
    users_events_by_time,
    on="visitorid",
    how="left",
).fillna(0)

In [28]:
# Посмотрим на результат
users_data.sort_values("adds_in_time_3-7", ascending=False).head(3)

,visitorid,adds_count,trans_cout,views_count,min_items_in_trans,mean_items_in_trans,max_items_in_trans,adds_in_day_0,trans_in_day_0,views_in_day_0,...,views_in_time_16-22,adds_in_time_22-3,trans_in_time_22-3,views_in_time_22-3,adds_in_time_3-7,trans_in_time_3-7,views_in_time_3-7,adds_in_time_7-12,trans_in_time_7-12,views_in_time_7-12
713559,1032742,0,0,0,0.0,0.0,0.0,0,0,0,...,13,21,0,42,48,0,129,0,0,0
169156,244620,0,0,0,0.0,0.0,0.0,0,0,0,...,0,0,0,0,44,0,12,0,0,0
249635,360921,0,0,0,0.0,0.0,0.0,7,0,31,...,1,5,0,12,33,0,86,0,0,0


In [29]:
display_nan_count(users_data)

**Количество пропусков: 0**

### Количество купленных товаров с `available == 0`

In [30]:
mask_available_0 = events_train_data["available"] == 0

users_transactions_with_available_0 = events_train_data[mask_available_0].pivot_table(
    index="visitorid",
    values="is_transaction",
    aggfunc="sum"
).reset_index() \
.rename(columns={"is_transaction": "trans_available_0"})

users_transactions_with_available_0.sort_values("trans_available_0", ascending=False).head(3)

,visitorid,trans_available_0
341029,1150086,45
22759,76757,16
224234,757355,15


In [31]:
# Объединим полученные данные с основной таблицей о пользователях
users_data = users_data.merge(
    users_transactions_with_available_0,
    on="visitorid",
    how="left",
).fillna(0)

# Посмотрим на результат
users_data.sort_values("trans_available_0", ascending=False).head(3)

,visitorid,adds_count,trans_cout,views_count,min_items_in_trans,mean_items_in_trans,max_items_in_trans,adds_in_day_0,trans_in_day_0,views_in_day_0,...,adds_in_time_22-3,trans_in_time_22-3,views_in_time_22-3,adds_in_time_3-7,trans_in_time_3-7,views_in_time_3-7,adds_in_time_7-12,trans_in_time_7-12,views_in_time_7-12,trans_available_0
794596,1150086,0,390,0,1.0,1.114286,5.0,61,51,687,...,146,131,1433,0,0,1,0,0,0,45.0
53197,76757,0,185,0,1.0,1.193548,5.0,73,36,274,...,65,42,365,0,0,0,0,0,0,16.0
95383,138131,0,173,0,1.0,1.262774,4.0,42,34,231,...,78,75,402,0,0,3,0,0,0,15.0


In [32]:
display_nan_count(users_data)

**Количество пропусков: 0**

### Средняя и медианная стоимость всех купленных товаров (`price`)

In [33]:
mask_transaction = events_train_data["is_transaction"] == True

user_check_agg = events_train_data[mask_transaction].pivot_table(
    index="visitorid",
    values="price",
    aggfunc=["mean", "median"]
)

# Переименуем колонки с мульти-индексом
user_check_agg.columns = user_check_agg.columns.to_flat_index()

user_check_agg = user_check_agg.reset_index().rename(columns={
    ("mean", "price"): "mean_check",
    ("median", "price"): "median_check",
})

user_check_agg.sort_values("median_check", ascending=False).head(3)

,visitorid,mean_check,median_check
6301,1129594,2765880.0,2765880.0
1925,346615,2279880.0,2279880.0
1068,201024,1792680.0,1792680.0


In [34]:
# Объединим полученные данные с основной таблицей о пользователях
users_data = users_data.merge(
    user_check_agg,
    on="visitorid",
    how="left",
).fillna(0)

# Посмотрим на результат
users_data.sort_values("median_check", ascending=False).head(3)

,visitorid,adds_count,trans_cout,views_count,min_items_in_trans,mean_items_in_trans,max_items_in_trans,adds_in_day_0,trans_in_day_0,views_in_day_0,...,views_in_time_22-3,adds_in_time_3-7,trans_in_time_3-7,views_in_time_3-7,adds_in_time_7-12,trans_in_time_7-12,views_in_time_7-12,trans_available_0,mean_check,median_check
780336,1129594,0,2,0,1.0,1.0,1.0,4,2,2,...,4,0,0,0,0,0,0,0.0,2765880.0,2765880.0
239748,346615,0,1,0,1.0,1.0,1.0,0,0,0,...,0,0,0,0,0,0,0,0.0,2279880.0,2279880.0
138880,201024,0,1,0,1.0,1.0,1.0,0,0,0,...,0,1,1,2,0,0,0,0.0,1792680.0,1792680.0


In [35]:
display_nan_count(users_data)

**Количество пропусков: 0**

### Количество купленных товаров с ценой 0 (`is_price_0`)

In [36]:
users_transactions_with_price_0 = events_train_data[mask_transaction].pivot_table(
    index="visitorid",
    values="is_price_0",
    aggfunc="sum",
).reset_index() \
.rename(columns={"is_price_0": "trans_price_0"})

users_transactions_with_price_0.sort_values("trans_price_0", ascending=False).head(3)

,visitorid,trans_price_0
6607,1150086,2
7579,1316484,1
6276,1096542,1


In [37]:
# Объединим полученные данные с основной таблицей о пользователях
users_data = users_data.merge(
    users_transactions_with_price_0,
    on="visitorid",
    how="left",
).fillna(0)

# Посмотрим на результат
users_data.sort_values("trans_price_0", ascending=False).head(3)

,visitorid,adds_count,trans_cout,views_count,min_items_in_trans,mean_items_in_trans,max_items_in_trans,adds_in_day_0,trans_in_day_0,views_in_day_0,...,adds_in_time_3-7,trans_in_time_3-7,views_in_time_3-7,adds_in_time_7-12,trans_in_time_7-12,views_in_time_7-12,trans_available_0,mean_check,median_check,trans_price_0
794596,1150086,0,390,0,1.0,1.114286,5.0,61,51,687,...,0,0,1,0,0,0,45.0,131615.679144,77940.0,2.0
909713,1316484,0,1,0,1.0,1.000000,1.0,0,0,0,...,0,0,0,0,0,0,1.0,0.000000,0.0,1.0
95383,138131,0,173,0,1.0,1.262774,4.0,42,34,231,...,0,0,3,0,0,0,15.0,113271.585366,51360.0,1.0


In [38]:
display_nan_count(users_data)

**Количество пропусков: 0**

### Количество купленных товаров в каждой корневой категории (`root_categoryid`)

In [39]:
users_tranactions_in_root_categories = events_train_data[mask_transaction].pivot_table(
    index="visitorid",
    columns="root_categoryid",
    values="is_transaction",
    aggfunc="sum",
    fill_value=0,
).reset_index()

# Переименуем колонки с номерами категорий
root_categories_column_names = list(users_tranactions_in_root_categories.columns)
root_categories_column_names.remove("visitorid")

columns_renaming = dict()

for cat in root_categories_column_names:
    columns_renaming[cat] = f"trans_in_cat_{int(cat)}"

users_tranactions_in_root_categories.rename(columns=columns_renaming, inplace=True)

users_tranactions_in_root_categories \
    .sort_values("trans_in_cat_1698", ascending=False) \
    .head(3)

root_categoryid,visitorid,trans_in_cat_140,trans_in_cat_250,trans_in_cat_378,trans_in_cat_395,trans_in_cat_431,trans_in_cat_653,trans_in_cat_659,trans_in_cat_679,trans_in_cat_791,trans_in_cat_803,trans_in_cat_859,trans_in_cat_1224,trans_in_cat_1452,trans_in_cat_1482,trans_in_cat_1490,trans_in_cat_1532,trans_in_cat_1579,trans_in_cat_1600,trans_in_cat_1698
5412,1145382,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,5
3968,852251,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4
6419,1352334,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4


In [40]:
# Объединим полученные данные с основной таблицей о пользователях
users_data = users_data.merge(
    users_tranactions_in_root_categories,
    on="visitorid",
    how="left",
).fillna(0)

# Посмотрим на результат
users_data.sort_values("trans_in_cat_1698", ascending=False).head(3)

,visitorid,adds_count,trans_cout,views_count,min_items_in_trans,mean_items_in_trans,max_items_in_trans,adds_in_day_0,trans_in_day_0,views_in_day_0,...,trans_in_cat_803,trans_in_cat_859,trans_in_cat_1224,trans_in_cat_1452,trans_in_cat_1482,trans_in_cat_1490,trans_in_cat_1532,trans_in_cat_1579,trans_in_cat_1600,trans_in_cat_1698
791303,1145382,0,7,0,1.0,1.4,3.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
934374,1352334,0,12,0,3.0,3.0,3.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
588869,852251,0,4,0,4.0,4.0,4.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0


In [41]:
display_nan_count(users_data)

**Количество пропусков: 0**

In [42]:
print(f"Размерность таблицы с данными пользователей: {users_data.shape}")

Размерность таблицы с данными пользователей: (972459, 66)


Сохраним полученную таблицу.

In [43]:
# Если таблицу еще не сохраняли, то нужно вызвать эту строку
# users_data.to_csv(f"{DATA_PATH}/users_data.csv", index=False)

In [44]:
# Если таблицу уже сохранили, то можно вызвать эту строку, 
# чтобы не повторять действия выше по созданию users_data
# users_data = pd.read_csv(f"{DATA_PATH}/users_data.csv")

## Создание таблицы с данными товаров

Из всех трех таблиц (с событиями, каталогом и данными о товарах) \
сформируем таблицу с данными о товарах.

Создадим следующие признаки.

- `itemid` товара.
- Входит ли в топ-10 самых покупаемых.
- Сколько раз товар купили в каждый день недели.
- Сколько раз товар купили в каждое время дня.
- Цена товара.
- Флаги принадлежности ко всей вложенной цепочке категорий товаров.

### Исследование пересечения данных о товарах в разных таблицах

На самом деле мы имеем не полное соответствие в данных.

- В таблице событий (`events_advanced_data`) есть товары, \
которых нет в таблице c обновлением данных о товарах (`items_last_data`).

- И наоборот, в таблице с обновлением данных о товарах (`items_last_data`)\
есть товары, которые не встречаются в событиях (`events_advanced_data`).

- Также в тестовой выборке `events_test_data`  есть товары, \
которых нет в тренировочной `events_train_data`.

- Данные в таблице с категориями (`category_data`) \
тоже не полностью соответствуют данным, \
которые указаны в свойстве `property="categoryid"` в событиях (`events_advanced_data`).

Посмотрим, сколько подобных случаев мы имеем.

In [45]:
# Множество уникальных itemid из исходной таблицы с событиями
itemids_from_events = set(events_advanced_data["itemid"].unique())

print(f"Количество уникальных товаров в исходной таблице с событиями: {len(itemids_from_events)}")

Количество уникальных товаров в исходной таблице с событиями: 235061


In [46]:
# Множество уникальных itemid из исходной таблицы с данными об обновлениях товаров
itemids_from_items = set(items_last_data["itemid"].unique())

print(f"Количество уникальных товаров в исходной таблице items_last_data: {len(itemids_from_items)}")

Количество уникальных товаров в исходной таблице items_last_data: 417053


In [47]:
# Множество товаров, которые упоминаются с событиях, но их нет в items_last_data
itemids_from_events_not_in_items = itemids_from_events - itemids_from_items

print(
    "Количество товаров, которые есть в событиях, но отсутствуют в items_last_data:",
    len(itemids_from_events_not_in_items),
)
print(
    "Процент товаров, которые есть в событиях, но отсутствуют в items_last_data:",
    round(len(itemids_from_events_not_in_items) / len(itemids_from_events) * 100, 2),
    "%"
)

Количество товаров, которые есть в событиях, но отсутствуют в items_last_data: 49815
Процент товаров, которые есть в событиях, но отсутствуют в items_last_data: 21.19 %


> Поскольку мы имеем не полные данные о каталоге товаров, а только фрагмент из сведений об их обновлениях,\
можно предположить, что данны о таких товарах существовали в каталоге ранее, \
и просто не обновлялись в доступный нам период.

In [48]:
# Множество товаров, которые есть в items_last_data, но отсутствуют в событиях
itemids_from_items_not_in_events = itemids_from_items - itemids_from_events

print(
    "Количество товаров, которые есть в items_data, но отсутствуют в событиях:",
    len(itemids_from_items_not_in_events)
)
print(
    "Процент товаров, которые есть в items_data, но отсутствуют в событиях:",
    round(len(itemids_from_items_not_in_events) / len(itemids_from_items) * 100, 2),
    "%"
)

Количество товаров, которые есть в items_data, но отсутствуют в событиях: 231807
Процент товаров, которые есть в items_data, но отсутствуют в событиях: 55.58 %


> То есть, более половины товаров из `items_last_data` не встречаются в событиях.\
Можно предположить, что пользователи не добрались до их просмотра в каталоге \
за доступный нам период.

In [49]:
# Множество itemid товаров из тренировочной таблицы с событиями
itemids_in_train_events = set(events_train_data["itemid"].unique())
# Множество itemid товаров из тестовой таблицы с событиями
itemids_in_test_events = set(events_test_data["itemid"].unique())

print(
    "Количество товаров в тренировочной таблице с событиями:",
    len(itemids_in_train_events)
)
print(
    "Количество товаров в тестовой таблице с событиями:",
    len(itemids_in_test_events)
)

Количество товаров в тренировочной таблице с событиями: 200456
Количество товаров в тестовой таблице с событиями: 141348


In [50]:
# Множество товаров, которые есть в тренировочной таблице с событиями и отсутствуют в тестовой
itemids_in_train_events_not_test = itemids_in_train_events - itemids_in_test_events

print(
    "Количество товаров, которые есть в тренировочной таблице событий и отсутствуют в тестовой:",
    len(itemids_in_train_events_not_test)
)
print(
    "Процент товаров, которые есть в тренировочной таблице событий и отсутствуют в тестовой:",
    round(len(itemids_in_train_events_not_test) / len(itemids_in_train_events) * 100, 2)
)

Количество товаров, которые есть в тренировочной таблице событий и отсутствуют в тестовой: 93713
Процент товаров, которые есть в тренировочной таблице событий и отсутствуют в тестовой: 46.75


In [51]:
# Множество товаров, которые есть в тестовой таблице с событиями и отсутствуют в тренировочной
itemids_in_test_events_not_train = itemids_in_test_events - itemids_in_train_events

print(
    "Количество товаров, которые есть в тестовой таблице событий и отсутствуют в тренировочной:",
    len(itemids_in_test_events_not_train)
)
print(
    "Процент товаров, которые есть в тестовой таблице событий и отсутствуют в тренировочной:",
    round(len(itemids_in_test_events_not_train) / len(itemids_in_test_events) * 100, 2)
)

Количество товаров, которые есть в тестовой таблице событий и отсутствуют в тренировочной: 34605
Процент товаров, которые есть в тестовой таблице событий и отсутствуют в тренировочной: 24.48


In [52]:
# Множество категорий из таблицы category_data
category_ids_from_catalog = set(category_data["categoryid"].unique())
category_ids_from_catalog = set(int(categoryid) for categoryid in category_ids_from_catalog)

# Множество категорий из таблицы items_data
mask_items_category = items_data["property"] == "categoryid"
category_ids_from_items = set(items_data[mask_items_category]["value"].unique())
category_ids_from_items = set(int(categoryid) for categoryid in category_ids_from_items)

print(
    "Количество уникальных категорий в каталоге category_data:", 
    len(category_ids_from_catalog)
)
print(
    "Количество уникальных категорий в таблице items_data:", 
    len(category_ids_from_items)
)

Количество уникальных категорий в каталоге category_data: 1669
Количество уникальных категорий в таблице items_data: 1242


In [53]:
# Категории, которые есть  в items_data и отсутствуют в каталоге
category_ids_from_items_not_catalog = category_ids_from_items - category_ids_from_catalog

print(
    "Количество категорий, которые есть в items_data и отсутствуют в каталоге:",
    len(category_ids_from_items_not_catalog)
)
print(
    "Процент категорий, которые есть в items_data и отсутствуют в каталоге:",
    round(len(category_ids_from_items_not_catalog) / len(category_ids_from_items) * 100, 2),
    "%"
)

Количество категорий, которые есть в items_data и отсутствуют в каталоге: 30
Процент категорий, которые есть в items_data и отсутствуют в каталоге: 2.42 %


> Таких категорий не так много, и это хорошо.

Далее соберем таблицу с данными о товарах.

### Входит ли в топ-10 самых покупаемых

Соберем сначала уникальные `itemid` всех товаров в одну таблицу.\
Возможно, потом для обучения модели мы отфильтруем какие-то товары.\
Но пока будем собирать данные по всем.

In [54]:
itemids_from_events = set(events_train_data["itemid"].unique())
itemids_from_items = set(items_last_data["itemid"].unique())

all_itemids = itemids_from_events | itemids_from_items

items_unique_data = pd.DataFrame({
    "itemid": list(all_itemids)
})

items_unique_data.head(3)

,itemid
0,0
1,1
2,2


In [55]:
# Выделим itemid самых популярных 10 товаров

MOST_POPULAR_ITEMS_COUNT = 10

mask_event_transaction = events_train_data["is_transaction"] == True

top_popular_itemids = events_train_data[mask_event_transaction]["itemid"] \
    .value_counts()[:MOST_POPULAR_ITEMS_COUNT].index

In [56]:
# Добавим признак с флагом популярности
items_unique_data["is_top_10"] = items_unique_data["itemid"].isin(top_popular_itemids)

items_unique_data.head(3)

,itemid,is_top_10
0,0,False
1,1,False
2,2,False


### Сколько раз товар купили в каждый день недели

In [57]:
items_trans_by_days = events_train_data.pivot_table(
    index="itemid",
    columns="day_of_week",
    values="is_transaction",
    aggfunc="sum",
    fill_value=0,
).reset_index()

# Переименуем колонки
columns_renaming = dict()
for day in range(0, 7):
    columns_renaming[day] = f"trans_in_day_{day}"

items_trans_by_days.rename(columns=columns_renaming, inplace=True)

items_trans_by_days.head(3)

day_of_week,itemid,trans_in_day_0,trans_in_day_1,trans_in_day_2,trans_in_day_3,trans_in_day_4,trans_in_day_5,trans_in_day_6
0,4,0,0,0,0,0,0,0
1,6,0,0,0,0,0,0,0
2,9,0,0,0,0,0,0,0


In [58]:
# Соединим данные с основной таблицей
items_unique_data = items_unique_data.merge(
    items_trans_by_days,
    how="left",
    on="itemid"
).fillna(0)

items_unique_data.head(3)

,itemid,is_top_10,trans_in_day_0,trans_in_day_1,trans_in_day_2,trans_in_day_3,trans_in_day_4,trans_in_day_5,trans_in_day_6
0,0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [59]:
display_nan_count(items_unique_data)

**Количество пропусков: 0**

### Сколько раз товар купили в каждое время дня

In [60]:
items_trans_by_time = events_train_data.pivot_table(
    index="itemid",
    columns="time_of_day",
    values="is_transaction",
    aggfunc="sum",
    fill_value=0,
).reset_index()

# Соберем словарь для переименования колонок
columns_renaming = dict()
for time in ["3-7", "7-12", "12-16", "16-22", "22-3"]:
    columns_renaming[time] = f"trans_in_time_{time}"

items_trans_by_time.rename(columns=columns_renaming, inplace=True)

items_trans_by_time.head(3)

time_of_day,itemid,trans_in_time_12-16,trans_in_time_16-22,trans_in_time_22-3,trans_in_time_3-7,trans_in_time_7-12
0,4,0,0,0,0,0
1,6,0,0,0,0,0
2,9,0,0,0,0,0


In [61]:
# Соединим данные с основной таблицей
items_unique_data = items_unique_data.merge(
    items_trans_by_time,
    how="left",
    on="itemid"
).fillna(0)

items_unique_data.head(3)

,itemid,is_top_10,trans_in_day_0,trans_in_day_1,trans_in_day_2,trans_in_day_3,trans_in_day_4,trans_in_day_5,trans_in_day_6,trans_in_time_12-16,trans_in_time_16-22,trans_in_time_22-3,trans_in_time_3-7,trans_in_time_7-12
0,0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [62]:
display_nan_count(items_unique_data)

**Количество пропусков: 0**

### Цена товара

In [63]:
# Как мы ранее определили, предположительно свойство цены лежит в property с именем "790"
mask_items_price = items_last_data["property"] == "790"

# Соберем словарь, где ключ - itemid товара, а значение - цена
items_prices = items_last_data[mask_items_price].set_index("itemid")["value"]
items_prices = { itemid: float(price[1:]) for itemid, price in items_prices.items()} 

# Заполним колонку с ценой
items_unique_data["last_price"] = items_unique_data["itemid"].apply(
    lambda itemid: items_prices[itemid] if itemid in items_prices else None
)
items_unique_data.head(3)

,itemid,is_top_10,trans_in_day_0,trans_in_day_1,trans_in_day_2,trans_in_day_3,trans_in_day_4,trans_in_day_5,trans_in_day_6,trans_in_time_12-16,trans_in_time_16-22,trans_in_time_22-3,trans_in_time_3-7,trans_in_time_7-12,last_price
0,0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,91200.0
1,1,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6120.0
2,2,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41040.0


In [64]:
# Посмотрим на долю пропусков в столбце
items_unique_data["last_price"].isna().mean()

0.08781258133731117

> Пропусков не так много, примерно 8%. \
Позже решим, как с ними поступим.

### Флаги принадлежности ко всей вложенной цепочке категорий товаров

In [65]:
# Сначала добавим признак с номером последнего уровня категории из таблицы items_last_data

mask_items_category = items_last_data["property"] == "categoryid"

# Соберем словарь со значениями категорий
items_categories = items_last_data[mask_items_category].set_index("itemid")["value"]
items_categories = { itemid: int(categoryid) for itemid, categoryid in items_categories.items()} 

# Создадим колонку со значением категории товара
items_unique_data["last_cat"] = items_unique_data["itemid"].apply(
    lambda itemid: items_categories[itemid] if itemid in items_categories else None
)

items_unique_data.head(3)

,itemid,is_top_10,trans_in_day_0,trans_in_day_1,trans_in_day_2,trans_in_day_3,trans_in_day_4,trans_in_day_5,trans_in_day_6,trans_in_time_12-16,trans_in_time_16-22,trans_in_time_22-3,trans_in_time_3-7,trans_in_time_7-12,last_price,last_cat
0,0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,91200.0,209.0
1,1,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6120.0,1114.0
2,2,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41040.0,1305.0


In [66]:
# Посмотрим на долю пропусков в столбце
items_unique_data["last_cat"].isna().mean()

0.08781258133731117

> Тоже примерно 8% пропусков. \
Позже решим, как с ними поступим.

Далее создадим признак со списком вложенных категорий, в которые входит товар.

In [67]:
# Создадим словарь из таблицы с категориями, 
# т.к. по нему будет быстрее происходить поиск, чем по таблице
catid_to_parentid = category_data.set_index("categoryid")["parentid"].to_dict()

# Сформируем признак в виде цепочки категорий (от корневой до текущей)
items_unique_data["cats_path"] = items_unique_data["last_cat"].apply(
    lambda last_cat: get_categories_levels_path(catid_to_parentid, last_cat) 
)

items_unique_data.head(3)

,itemid,is_top_10,trans_in_day_0,trans_in_day_1,trans_in_day_2,trans_in_day_3,trans_in_day_4,trans_in_day_5,trans_in_day_6,trans_in_time_12-16,trans_in_time_16-22,trans_in_time_22-3,trans_in_time_3-7,trans_in_time_7-12,last_price,last_cat,cats_path
0,0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,91200.0,209.0,"[1532, 293, 209]"
1,1,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6120.0,1114.0,"[1532, 113, 1114]"
2,2,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41040.0,1305.0,"[653, 1202, 1214, 1305]"


> Здесь нужно учитывать, что категории для колонки `last_cat` мы брали из таблицы\
с обновлением товаров (`items_last_data`).\
И как мы ранее выяснили, не все из них есть в таблице категорий `category_data`.\
Поэтому в некоторых случаях в колонке `cats_path` мы получим\
список с единственной категорией. И не потому, что она рутовая (корневая),\
а потому, что информации о ней просто нет в `category_data`.\
Но мы все равно создадим признак и из таких одиночных категорий.

Далее для каждой категории создадим свой столбец.\
И проставим 1, если товар относится к ней и 0, если не относится.\
Это даст нам признаки, которые позволят составить относительно точные векторы,\
по которым можно будет определить схожие товары.

In [87]:
# Выделим множество категорий, которые есть у товаров
cats_ids = set(items_unique_data["last_cat"].unique())
cats_ids = {int(cat_id) for cat_id in cats_ids if not np.isnan(cat_id)}

for cat_id in cats_ids:
    cat_column_name = f"cat_{cat_id}"
    
    items_unique_data[cat_column_name] = items_unique_data["cats_path"].apply(
        lambda cats_path: 1 if cat_id in cats_path else 0
    )

In [88]:
items_unique_data.head(3)

,itemid,is_top_10,trans_in_day_0,trans_in_day_1,trans_in_day_2,trans_in_day_3,trans_in_day_4,trans_in_day_5,trans_in_day_6,trans_in_time_12-16,...,cat_1680,cat_1681,cat_1684,cat_1685,cat_1686,cat_1689,cat_1690,cat_1694,cat_1695,cat_1697
0,0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,1,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,2,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [89]:
print(
    "Размерность таблицы items_unique_data с товарами:",
    items_unique_data.shape
)

Размерность таблицы items_unique_data с товарами: (457201, 1197)


Сохраним полученную таблицу.

In [ ]:
# Если таблицу еще не сохраняли, то нужно вызвать эту строку
# items_unique_data.to_csv(f"{DATA_PATH}/items_data.csv", index=False)

In [ ]:
# Если таблицу уже сохранили, то можно вызвать эту строку, 
# чтобы не повторять действия выше по созданию items_unique_data
# items_unique_data = pd.read_csv(f"{DATA_PATH}/items_data.csv")